In [ ]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
#from langchain.text_splitter import RecursiveCharacterTextSplitter 
from pathlib import Path

In [ ]:
folder_path = r"G:\langchain project wit RAG\data\raw"

# Validate folder
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# Get PDF files (case-insensitive)
pdf_files = [
    file for file in os.listdir(folder_path)
    if file.lower().endswith(".pdf")
]

# Check if folder is empty
if not pdf_files:
    raise ValueError("No PDF files found in folder.")

all_documents = []
failed_files = []

for pdf in pdf_files:
    file_path = os.path.join(folder_path, pdf)

    try:
        loader = PyPDFLoader(file_path)
        documents = loader.load()

        # Add metadata
        for doc in documents:
            doc.metadata["source_file"] = pdf
            doc.metadata["full_path"] = file_path

        all_documents.extend(documents)

        print(f"✅ Loaded: {pdf}")
        print(f"Pages: {len(documents)}")
        print("-" * 50)

    except Exception as e:
        failed_files.append(pdf)
        print(f"❌ Failed: {pdf}")
        print(f"Error: {e}")
        print("-" * 50)

print("\n========== SUMMARY ==========")
print(f"Total PDFs Found: {len(pdf_files)}")
print(f"Successfully Loaded: {len(pdf_files) - len(failed_files)}")
print(f"Failed Files: {len(failed_files)}")
print(f"Total Documents Loaded: {len(all_documents)}")

if failed_files:
    print("\nFailed PDFs:")
    for file in failed_files:
        print(file)

In [ ]:
all_documents

In [ ]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [ ]:
chunks = split_documents(all_documents)
chunks

### Chunking


In [ ]:
print(f"Chunks: {len(chunks)}")


# DAY2

### embedding And vectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import (
    SentenceTransformer)
from typing import List
import numpy as np
import torch
import time
import os


In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

from typing import List
import numpy as np
import torch
import time
import os


class EmbeddingManager:
    """
    Production-grade embedding manager
    Optimized for CPU + 16GB RAM
    """

    def __init__(
        self,
        model_name="BAAI/bge-base-en-v1.5",
        device="cpu",
        cache_dir="cache"
    ):

        self.model_name = model_name
        self.device = device
        self.cache_dir = cache_dir

        os.makedirs(
            self.cache_dir,
            exist_ok=True
        )

        self.model = None

        self._load_model()

    # ==========================
    # Load model
    # ==========================
    def _load_model(self):

        try:

            print(
                f"Loading model: "
                f"{self.model_name}"
            )

            start = time.time()

            self.model = SentenceTransformer(
                self.model_name,
                device=self.device
            )

            elapsed = (
                time.time() - start
            )

            print(
                f"Loaded on "
                f"{self.device}"
            )

            print(
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

            print(
                f"Model loaded in "
                f"{elapsed:.2f}s"
            )

        except Exception as e:
            print(
                f"Model loading failed: "
                f"{e}"
            )
            raise

    # ==========================
    # Clean text
    # ==========================
    def _clean_texts(
        self,
        texts: List[str]
    ) -> List[str]:

        cleaned = []

        for text in texts:

            if not text:
                continue

            text = (
                text
                .replace("\n", " ")
                .strip()
            )

            text = " ".join(
                text.split()
            )

            # Skip junk chunks
            if len(text) < 20:
                continue

            cleaned.append(text)

        return cleaned

    # ==========================
    # Embed documents
    # ==========================
import hashlib

def embed_documents(self, texts: List[str], batch_size=8, save_cache=True) -> np.ndarray:
    if not texts:
        raise ValueError("Text list is empty")

    texts = self._clean_texts(texts)
    if not texts:
        raise ValueError("No valid texts found")

    # Build a cache key from the actual content + length
    content_hash = hashlib.md5(
        ("".join(texts) + str(len(texts))).encode("utf-8")
    ).hexdigest()[:10]

    cache_file = os.path.join(
        self.cache_dir, f"embeddings_{len(texts)}_{content_hash}.npy"
    )

    if save_cache and os.path.exists(cache_file):
        print("Loading cached embeddings...")
        embeddings = np.load(cache_file)
        print(f"Loaded cached shape: {embeddings.shape}")
        return embeddings

    print(f"Generating embeddings for {len(texts)} texts...")
    start = time.time()

    texts_prefixed = [f"passage: {t}" for t in texts]

    embeddings = self.model.encode(
        texts_prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    elapsed = time.time() - start

    if save_cache:
        np.save(cache_file, embeddings)
        print(f"Saved cache → {cache_file}")

    print("=" * 50)
    print("EMBEDDING SUMMARY")
    print(f"Texts embedded: {len(texts)}")
    print(f"Shape: {embeddings.shape}")
    print(f"Time taken: {elapsed:.2f}s")
    print(f"Avg/sec: {len(texts)/elapsed:.2f}")
    print("=" * 50)

    return embeddings

    # ==========================
    # Embed query
    # ==========================
def embed_query(
        self,
        query: str
    ) -> np.ndarray:

        if not query.strip():
            raise ValueError(
                "Query is empty"
            )

        query = (
            f"query: "
            f"{query.strip()}"
        )

        embedding = self.model.encode(
            query,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        return embedding

In [ ]:
import hashlib
import os
import time
from typing import List

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """
    Production-grade embedding manager
    Optimized for CPU + 16GB RAM
    """

    def __init__(
        self,
        model_name="BAAI/bge-base-en-v1.5",
        device="cpu",
        cache_dir="cache"
    ):
        self.model_name = model_name
        self.device = device
        self.cache_dir = cache_dir

        os.makedirs(self.cache_dir, exist_ok=True)

        self.model = None
        self._load_model()

    # ==========================
    # Load model
    # ==========================
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            start = time.time()

            self.model = SentenceTransformer(
                self.model_name,
                device=self.device
            )

            elapsed = time.time() - start

            print(f"Loaded on {self.device}")
            print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
            print(f"Model loaded in {elapsed:.2f}s")

        except Exception as e:
            print(f"Model loading failed: {e}")
            raise

    # ==========================
    # Clean text
    # ==========================
    def _clean_texts(self, texts: List[str]) -> List[str]:
        cleaned = []

        for text in texts:
            if not text:
                continue

            text = text.replace("\n", " ").strip()
            text = " ".join(text.split())

            # Skip junk chunks
            if len(text) < 20:
                continue

            cleaned.append(text)

        return cleaned

    # ==========================
    # Embed documents
    # ==========================
    def embed_documents(
        self,
        texts: List[str],
        batch_size=8,
        save_cache=True
    ) -> np.ndarray:

        if not texts:
            raise ValueError("Text list is empty")

        texts = self._clean_texts(texts)

        if not texts:
            raise ValueError("No valid texts found")

        # Content-aware cache key, so stale caches from older/smaller
        # document sets never get returned for a different text list.
        content_hash = hashlib.md5(
            ("".join(texts) + str(len(texts))).encode("utf-8")
        ).hexdigest()[:10]

        cache_file = os.path.join(
            self.cache_dir, f"embeddings_{len(texts)}_{content_hash}.npy"
        )

        # ----------------------
        # Load cache
        # ----------------------
        if save_cache and os.path.exists(cache_file):
            print("Loading cached embeddings...")
            embeddings = np.load(cache_file)
            print(f"Loaded cached shape: {embeddings.shape}")
            return embeddings

        print(f"Generating embeddings for {len(texts)} texts...")
        start = time.time()

        # Important for BGE retrieval
        texts_prefixed = [f"passage: {text}" for text in texts]

        embeddings = self.model.encode(
            texts_prefixed,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        elapsed = time.time() - start

        # ----------------------
        # Save cache
        # ----------------------
        if save_cache:
            np.save(cache_file, embeddings)
            print(f"Saved cache → {cache_file}")

        print("=" * 50)
        print("EMBEDDING SUMMARY")
        print("=" * 50)
        print(f"Texts embedded: {len(texts)}")
        print(f"Shape: {embeddings.shape}")
        print(f"Time taken: {elapsed:.2f}s")
        print(f"Avg/sec: {len(texts)/elapsed:.2f}")
        print("=" * 50)

        return embeddings

    # ==========================
    # Embed query
    # ==========================
    def embed_query(self, query: str) -> np.ndarray:
        if not query.strip():
            raise ValueError("Query is empty")

        query = f"query: {query.strip()}"

        embedding = self.model.encode(
            query,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        return embedding

In [ ]:
embedding_manager=EmbeddingManager()
embedding_manager


In [ ]:
embedding_manager = EmbeddingManager()
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.embed_documents(texts)
print(type(embeddings), embeddings.shape, len(texts))

In [ ]:
print(f"Chunks: {len(chunks)}")
print(f"Texts: {len(texts)}")

### VectorStore


In [ ]:
import os
import gc
import time
import hashlib
import chromadb
import numpy as np

from typing import List, Any, Optional


class VectorStore:
    """
    Production-grade ChromaDB manager
    """

    def __init__(
        self,
        collection_name="pdf_documents_bge_base_v1",
        persist_directory="../data/vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = (
            persist_directory
        )

        self.client = None
        self.collection = None

        self._initialize_store()

    # ==========================
    # Initialize
    # ==========================
    def _initialize_store(self):

        try:

            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            self.client = (
                chromadb.PersistentClient(
                    path=self.persist_directory
                )
            )

            self.collection = (
                self.client
                .get_or_create_collection(
                    name=self.collection_name,
                    metadata={
                        "description":
                        "Production RAG store"
                    }
                )
            )

            print("=" * 60)
            print("VECTOR STORE INITIALIZED")
            print("=" * 60)

            print(
                f"Collection: "
                f"{self.collection_name}"
            )

            print(
                f"Existing docs: "
                f"{self.collection.count()}"
            )

            print("=" * 60)

        except Exception as e:

            print(
                f"Error initializing "
                f"vector store: {e}"
            )

            raise

    # ==========================
    # Clean metadata
    # ==========================
    def _clean_metadata(
        self,
        metadata
    ):

        cleaned = {}

        for k, v in metadata.items():

            if isinstance(
                v,
                (str, int, float, bool)
            ):
                cleaned[k] = v

            elif v is None:
                cleaned[k] = "None"

            else:
                cleaned[k] = str(v)

        return cleaned

    # ==========================
    # Validate embedding dim
    # ==========================
    def _validate_embedding_dimension(
        self,
        embeddings
    ):

        embedding_dim = len(
            embeddings[0]
        )

        if self.collection.count() > 0:

            sample = (
                self.collection.peek(
                    limit=1
                )
            )

            if (
                "embeddings" in sample
                and sample["embeddings"] is not None
                and len(sample["embeddings"]) > 0
                ):

                existing_dim = len(
                    sample[
                        "embeddings"
                    ][0]
                )

                if (
                    existing_dim
                    != embedding_dim
                ):
                    raise ValueError(
                        f"Embedding "
                        f"dimension mismatch. "
                        f"Expected "
                        f"{existing_dim}, "
                        f"got "
                        f"{embedding_dim}"
                    )

    # ==========================
    # Add docs
    # ==========================
    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray,
        batch_size=200
    ):

        if len(documents) != len(
            embeddings
        ):
            raise ValueError(
                "Documents and embeddings "
                "count mismatch"
            )

        self._validate_embedding_dimension(
            embeddings
        )

        print(
            f"Adding "
            f"{len(documents)} chunks..."
        )

        start_time = time.time()

        total_added = 0
        duplicate_count = 0

        for batch_start in range(
            0,
            len(documents),
            batch_size
        ):

            batch_docs = documents[
                batch_start:
                batch_start
                + batch_size
            ]

            batch_embeddings = embeddings[
                batch_start:
                batch_start
                + batch_size
            ]

            ids = []
            metadatas = []
            texts = []
            embed_list = []

            for doc, emb in zip(
                batch_docs,
                batch_embeddings
            ):

                content = (
                    doc.page_content
                )

                chunk_id = (
                    doc.metadata.get(
                        "chunk_id"
                    )
                )

                # Stable hash ID
                if not chunk_id:

                    content_hash = (
                        hashlib.md5(
                            content.encode()
                        )
                        .hexdigest()
                    )

                    chunk_id = (
                        f"chunk_"
                        f"{content_hash}"
                    )

                # Prevent duplicates
                existing = (
                    self.collection.get(
                        ids=[chunk_id]
                    )
                )

                if (
                    existing
                    and existing["ids"]
                ):
                    duplicate_count += 1
                    continue

                ids.append(chunk_id)

                metadata = (
                    self._clean_metadata(
                        doc.metadata
                    )
                )

                metadata.update({
                    "content_length":
                    len(content)
                })

                metadatas.append(
                    metadata
                )

                texts.append(content)

                embed_list.append(
                    emb.tolist()
                )

            if not ids:
                continue

            try:

                self.collection.upsert(
                    ids=ids,
                    embeddings=
                    embed_list,
                    metadatas=
                    metadatas,
                    documents=texts
                )

                total_added += len(ids)

                print(
                    f"Added "
                    f"{total_added}/"
                    f"{len(documents)}"
                )

            except Exception as e:

                print(
                    f"Batch failed: "
                    f"{e}"
                )

            # CPU memory cleanup
            gc.collect()

        elapsed = (
            time.time()
            - start_time
        )

        print("=" * 60)
        print("VECTOR STORE SUMMARY")
        print("=" * 60)

        print(
            f"Collection: "
            f"{self.collection_name}"
        )

        print(
            f"Added docs: "
            f"{total_added}"
        )

        print(
            f"Duplicates skipped: "
            f"{duplicate_count}"
        )

        print(
            f"Total docs: "
            f"{self.collection.count()}"
        )

        print(
            f"Time: "
            f"{elapsed:.2f}s"
        )

        print("=" * 60)

    # ==========================
    # Search
    # ==========================
    def search(
        self,
        query_embedding,
        top_k=5,
        where=None
    ):

        if query_embedding is None:
            raise ValueError(
                "Query embedding missing"
            )

        if (
            not isinstance(
                query_embedding,
                np.ndarray
            )
        ):
            raise TypeError(
                "Query embedding "
                "must be numpy array"
            )

        start = time.time()

        results = (
            self.collection.query(
                query_embeddings=[
                    query_embedding
                    .tolist()
                ],
                n_results=top_k,
                where=where,
                include=[
                    "documents",
                    "metadatas",
                    "distances"
                ]
            )
        )

        formatted_results = []

        for i in range(
            len(
                results[
                    "documents"
                ][0]
            )
        ):

            formatted_results.append({
                "content":
                results[
                    "documents"
                ][0][i],

                "metadata":
                results[
                    "metadatas"
                ][0][i],

                "score":
                results[
                    "distances"
                ][0][i]
            })

        elapsed = (
            time.time()
            - start
        )

        print(
            f"Search done "
            f"in {elapsed:.2f}s"
        )

        return formatted_results

    # ==========================
    # Retrieve context
    # ==========================
    def retrieve_context(
        self,
        query_embedding,
        top_k=5,
        where=None
    ):

        results = self.search(
            query_embedding,
            top_k=top_k,
            where=where
        )

        context = "\n\n".join(
            r["content"]
            for r in results
        )

        return context

    # ==========================
    # Reset collection
    # ==========================
    def reset_collection(
        self
    ):

        self.client.delete_collection(
            self.collection_name
        )

        self._initialize_store()

        gc.collect()

        print(
            "Collection reset done."
        )

In [ ]:
vectorstore=VectorStore()
vectorstore

In [ ]:
import time

# ==========================
# Extract text
# ==========================
texts = [
    doc.page_content.strip()
    for doc in chunks
]

print(
    f"Preparing "
    f"{len(texts)} chunks..."
)

# ==========================
# Generate embeddings
# ==========================
start = time.time()

embeddings = (
    embedding_manager
    .embed_documents(
        texts=texts,
        batch_size=8,
        save_cache=True
    )
)

print(
    f"Embedding completed "
    f"in {time.time()-start:.2f}s"
)

# ==========================
# Validation
# ==========================
if len(chunks) != embeddings.shape[0]:

    raise ValueError(
        f"Chunk/Embedding mismatch: "
        f"{len(chunks)} chunks vs "
        f"{embeddings.shape[0]} embeddings"
    )

# ==========================
# Store in ChromaDB
# ==========================
vectorstore.add_documents(
    documents=chunks,
    embeddings=embeddings,
    batch_size=100
)

print(
    "\nPipeline completed successfully."
)

In [ ]:
print(vectorstore.collection.peek(limit=1))

In [ ]:
print(vectorstore.collection.count())

### Retriever Pipeline From VectorStore

In [ ]:
from typing import List, Dict, Any
from sentence_transformers import CrossEncoder

from rank_bm25 import BM25Okapi
from collections import defaultdict

import numpy as np
import time


class RAGRetriever:
    """
    Production Grade Hybrid Retriever

    Features
    --------
    1. Dense Retrieval (BGE)
    2. BM25 Retrieval
    3. Reciprocal Rank Fusion (RRF)
    4. CrossEncoder Re-ranking
    5. Query Cache
    6. Performance Monitoring
    """

    def __init__(
        self,
        vector_store,
        embedding_manager,
        documents,
        reranker_model="BAAI/bge-reranker-base"
    ):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

        self.documents = documents

        self.query_cache = {}

        print("=" * 60)
        print(
            f"Loading reranker: "
            f"{reranker_model}"
        )
        print("=" * 60)

        self.reranker = CrossEncoder(
            reranker_model
        )

        # ==========================================
        # STEP 1
        # Build BM25 Index
        # ==========================================

        print("=" * 60)
        print("Building BM25 Index...")
        print("=" * 60)

        self.bm25_corpus = [

            doc.page_content.split()

            for doc in documents

        ]

        self.bm25 = BM25Okapi(
            self.bm25_corpus
        )

        print(
            f"BM25 Indexed "
            f"{len(documents)} chunks"
        )

    # ==================================================
    # Cached Query Embedding
    # ==================================================

    def _embed_query(
        self,
        query: str
    ):

        if query in self.query_cache:
            return self.query_cache[query]

        embedding = (
            self.embedding_manager
            .embed_query(query)
        )

        self.query_cache[
            query
        ] = embedding

        return embedding

    # ==================================================
    # STEP 2
    # BM25 Retrieval
    # ==================================================

    def bm25_search(
        self,
        query,
        top_k=20
    ):

        tokens = query.split()

        scores = (
            self.bm25
            .get_scores(tokens)
        )

        ranked_idx = np.argsort(
            scores
        )[::-1]

        results = []

        for idx in ranked_idx[:top_k]:

            doc = self.documents[idx]

            results.append({

                "content":
                doc.page_content,

                "metadata":
                doc.metadata,

                "bm25_score":
                float(scores[idx])

            })

        return results

    # ==================================================
    # STEP 3
    # Reciprocal Rank Fusion
    # ==================================================

    def reciprocal_rank_fusion(
        self,
        dense_results,
        bm25_results,
        k=60
    ):

        scores = defaultdict(float)

        all_docs = {}

        # Dense Ranking

        for rank, doc in enumerate(
            dense_results
        ):

            doc_id = (

                f"{doc['metadata'].get('source_file','')}"
                f"_{doc['metadata'].get('page','')}"
                f"_{doc['metadata'].get('chunk_index','')}"

            )

            scores[doc_id] += (
                1 / (k + rank + 1)
            )

            all_docs[doc_id] = doc

        # BM25 Ranking

        for rank, doc in enumerate(
            bm25_results
        ):

            doc_id = (

                f"{doc['metadata'].get('source_file','')}"
                f"_{doc['metadata'].get('page','')}"
                f"_{doc['metadata'].get('chunk_index','')}"

            )

            scores[doc_id] += (
                1 / (k + rank + 1)
            )

            all_docs[doc_id] = doc

        fused = sorted(

            all_docs.items(),

            key=lambda x:
            scores[x[0]],

            reverse=True

        )

        fused_results = [

            doc

            for _, doc in fused

        ]

        return fused_results

    # ==================================================
    # STEP 4
    # Main Retrieval
    # ==================================================

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        retrieval_k: int = 50,
        score_threshold: float = -999,
        where: dict = None
    ) -> List[Dict[str, Any]]:

        total_start = time.time()

        query = " ".join(
            query.strip().split()
        )

        if not query:
            raise ValueError(
                "Query cannot be empty."
            )

        print("=" * 60)
        print("HYBRID RETRIEVAL")
        print("=" * 60)

        print(f"Query: {query}")

        # ======================================
        # Dense Retrieval
        # ======================================

        query_embedding = (
            self._embed_query(query)
        )

        dense_results = (

            self.vector_store.search(

                query_embedding=
                query_embedding,

                top_k=
                retrieval_k,

                where=
                where

            )

        )

        # ======================================
        # STEP 5
        # BM25 Retrieval
        # ======================================

        bm25_results = (

            self.bm25_search(

                query=query,

                top_k=retrieval_k

            )

        )

        # ======================================
        # RRF Fusion
        # ======================================

        results = (

            self.reciprocal_rank_fusion(

                dense_results,

                bm25_results

            )

        )

        if not results:
            return []

        # ======================================
        # CrossEncoder Re-ranking
        # ======================================

        pairs = [

            (
                query,
                r["content"]
            )

            for r in results[:30]

        ]

        rerank_scores = (

            self.reranker.predict(
                pairs
            )

        )

        reranked_results = []

        for result, score in zip(
            results[:30],
            rerank_scores
        ):

            metadata = result.get(
                "metadata",
                {}
            )

            reranked_results.append({

                "content":
                result["content"],

                "rerank_score":
                float(score),

                "metadata":
                metadata,

                "source_file":
                metadata.get(
                    "source_file",
                    "Unknown"
                ),

                "page":
                metadata.get(
                    "page",
                    "Unknown"
                ),

                "chunk_index":
                metadata.get(
                    "chunk_index",
                    "Unknown"
                )

            })

        reranked_results.sort(

            key=lambda x:
            x["rerank_score"],

            reverse=True

        )

        final_results = [

            r

            for r in reranked_results

            if (
                r["rerank_score"]
                >= score_threshold
            )

        ][:top_k]

        for rank, result in enumerate(
            final_results,
            start=1
        ):

            result["rank"] = rank

        total_time = (
            time.time()
            - total_start
        )

        print("=" * 60)

        print(
            f"Final Results: "
            f"{len(final_results)}"
        )

        print(
            f"Total Time: "
            f"{total_time:.2f}s"
        )

        print("=" * 60)

        return final_results

In [ ]:
RAGRetriever

In [ ]:
retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager,
    documents=chunks
)

In [ ]:
def print_results(results, query=None, max_chars=1200):
    """
    Pretty-print retrieval results from retriever.retrieve()

    Args:
        results: list of dicts (your retrieval output)
        query: optional query string to display as a header
        max_chars: how much of each chunk's content to show
    """
    print("=" * 70)
    if query:
        print(f"QUERY: {query}")
    print(f"RESULTS: {len(results)}")
    print("=" * 70)

    for r in results:
        meta = r.get("metadata", {})
        content = r.get("content", "").strip().replace("\n", " ")
        if len(content) > max_chars:
            content = content[:max_chars].rstrip() + " ..."

        print(f"\n[Rank {r.get('rank')}]  score: {r.get('rerank_score'):.3f}")
        print(f"  Source : {r.get('source_file')}")
        print(f"  Page   : {meta.get('page_label', r.get('page'))}")
        print(f"  Text   : {content}")
        print("-" * 70)


# ==========================
# Usage
# ==========================
#results = retriever.retrieve(query="Unified Multi-task Learning Framework")
#print_results(results, query="Unified Multi-task Learning Framework")
#results = retriever.retrieve(query="What datasets were used for machine translation experiments?")
#print_results(results, query="What datasets were used for machine translation experiments?")

results = retriever.retrieve(query="What are the limitations of Retrieval-Augmented Generation?")
print_results(results, query="What are the limitations of Retrieval-Augmented Generation?")


In [ ]:
def print_results(results, query=None, max_chars=300):
    """
    Pretty-print retrieval results from retriever.retrieve()
    """
    print("=" * 70)
    if query:
        print(f"QUERY: {query}")
    print(f"RESULTS: {len(results)}")
    print("=" * 70)

    for r in results:
        meta = r.get("metadata", {})
        content = r.get("content", "").strip().replace("\n", " ")
        if len(content) > max_chars:
            content = content[:max_chars].rstrip() + " ..."

        print(f"\n[Rank {r.get('rank')}]  score: {r.get('rerank_score'):.3f}")
        print(f"  Source : {r.get('source_file')}")
        print(f"  Page   : {meta.get('page_label', r.get('page'))}")
        print(f"  Text   : {content}")
        print("-" * 70)


def save_top_answers(query_results_list, output_path="query_answers.txt", max_chars=500):
    """
    Save each query along with its Rank 1 (top) retrieved chunk to a txt file.
    Appends to the file if it already exists, so previous queries are preserved.

    Args:
        query_results_list: list of (query, results) tuples
        output_path: path to the output .txt file
        max_chars: how much of the rank-1 content to include
    """
    with open(output_path, "a", encoding="utf-8") as f:
        for query, results in query_results_list:
            top = next((r for r in results if r.get("rank") == 1), results[0] if results else None)

            f.write("=" * 70 + "\n")
            f.write(f"QUERY: {query}\n")
            f.write("=" * 70 + "\n")

            if top is None:
                f.write("ANSWER: No results found.\n\n")
                continue

            content = top.get("content", "").strip().replace("\n", " ")
            if len(content) > max_chars:
                content = content[:max_chars].rstrip() + " ..."

            meta = top.get("metadata", {})
            f.write(f"ANSWER : {content}\n")
            f.write(f"SOURCE : {top.get('source_file')}\n")
            f.write(f"PAGE   : {meta.get('page_label', top.get('page'))}\n")
            f.write(f"SCORE  : {top.get('rerank_score'):.3f}\n\n")

    print(f"Saved {len(query_results_list)} query/answer pairs → {output_path}")


# ==========================
# Usage
# ==========================
queries = [ 
    "What are the advantages of RAG over parametric language models?",
    "Why does ReAct outperform Chain-of-Thought prompting?",
    "Why are Transformers faster to train than RNNs?",
    "Why do long-context models struggle with information in the middle of documents?",
    "What are the limitations of Retrieval-Augmented Generation?"
]

query_results_list = []

for q in queries:
    results = retriever.retrieve(query=q)
    print_results(results, query=q)        # keep console output as before
    query_results_list.append((q, results))

# Save everything to a txt file
save_top_answers(query_results_list, output_path="query_answers.txt")

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
import os

from langchain_groq import ChatGroq
from langchain_core.messages import (
    HumanMessage,
    SystemMessage
)


class GroqLLM:
    """
    Production-grade Groq LLM wrapper
    optimized for RAG.
    """

    def __init__(
        self,
        model_name="llama-3.3-70b-versatile",
        api_key=None,
        temperature=0,
        max_tokens=1024,
        max_context_chars=25000
    ):

        self.api_key = (
            api_key
            or os.getenv("GROQ_API_KEY")
        )

        if not self.api_key:
            raise ValueError(
                "GROQ_API_KEY not found."
            )

        self.model_name = model_name
        self.max_context_chars = (
            max_context_chars
        )

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )

        print("=" * 60)
        print("GROQ LLM INITIALIZED")
        print("=" * 60)
        print(f"Model: {self.model_name}")
        print(f"Temperature: {temperature}")
        print(f"Max Tokens: {max_tokens}")
        print("=" * 60)

    # =====================================
    # Build Context
    # =====================================

    def build_context(
        self,
        retrieval_results
    ) -> str:

        context_parts = []

        for result in retrieval_results:

            metadata = result.get(
                "metadata",
                {}
            )

            source_file = metadata.get(
                "source_file",
                "Unknown"
            )

            page = metadata.get(
                "page",
                "Unknown"
            )

            chunk = metadata.get(
                "chunk_index",
                "Unknown"
            )

            content = result.get(
                "content",
                ""
            )

            context_parts.append(
                f"""
CHUNK: {chunk}

{content}
"""
            )

        context = "\n\n".join(
            context_parts
        )

        # Context length protection

        if len(context) > self.max_context_chars:

            context = context[
                :self.max_context_chars
            ]

        return context

    # =====================================
    # Generate Answer
    # =====================================

    def generate_response(
        self,
        query: str,
        retrieval_results
    ):

        if not retrieval_results:

            return (
                "I could not find the answer "
                "in the provided documents."
            )

        context = self.build_context(
            retrieval_results
        )

        prompt = f"""
You are a Retrieval-Augmented AI Assistant.

Rules:

1. Use ONLY the provided context.

2. Do NOT use outside knowledge.

3. Do NOT invent information.

4. If the answer is not found in the context,
respond exactly with:

"I could not find the answer in the provided documents."

5. Do NOT mention source files,
page numbers, chunk numbers,
or retrieval metadata in the answer.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
"""

        try:

            messages = [

                SystemMessage(
                    content=
                    """
You are a professional
RAG assistant.

Use ONLY retrieved context.

Never hallucinate.
"""
                ),

                HumanMessage(
                    content=prompt
                )

            ]

            response = self.llm.invoke(
                messages
            )

            return response.content

        except Exception as e:

            return (
                f"Generation Error: {e}"
            )

    # =====================================
    # Full RAG Pipeline
    # =====================================

    def ask(
        self,
        query,
        retriever,
        top_k=5,
        score_threshold=0
    ):

        results = retriever.retrieve(
            query=query,
            top_k=top_k
        )

        # Filter weak chunks

        results = [

            r

            for r in results

            if r["rerank_score"]
            >= score_threshold

        ]

        answer = (
            self.generate_response(
                query=query,
                retrieval_results=results
            )
        )

        return {

            "query": query,

            "answer": answer,
        }

In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

llm = GroqLLM(
    model_name="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

print("LLM initialized successfully!")

In [ ]:
query = "Explain the complete RAG pipeline from retrieval to generation."

results = retriever.retrieve(query=query)

response = llm.generate_response(
    query=query,
    retrieval_results=results
)

print(response)

### Integration Vectordb Context pipeline With LLM output

In [ ]:
import numpy as np


def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0,
    return_context=False
):
    """
    Production-grade RAG Pipeline

    Returns:
        answer
        sources
        retrieval_score
        context (optional)
    """

    # ==========================================
    # Retrieve
    # ==========================================

    results = retriever.retrieve(
        query=query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:

        return {

            "answer":
            "No relevant context found.",

            "sources": [],

            "retrieval_score": 0.0
        }

    # ==========================================
    # Generate Answer
    # ==========================================

    answer = llm.generate_response(
        query=query,
        retrieval_results=results
    )

    # ==========================================
    # Build Source List
    # ==========================================

    seen = set()

    sources = []

    for doc in results:

        metadata = doc.get(
            "metadata",
            {}
        )

        source_file = metadata.get(
            "source_file",
            "unknown"
        )

        page = metadata.get(
            "page",
            "unknown"
        )

        key = (
            source_file,
            page
        )

        if key in seen:
            continue

        seen.add(key)

        sources.append({

            "source":
            source_file,

            "page":
            page,

            "score":
            round(
                doc.get(
                    "rerank_score",
                    0
                ),
                3
            )

        })

    # ==========================================
    # Retrieval Score
    # ==========================================

    retrieval_score = round(

        np.mean([

            r.get(
                "rerank_score",
                0
            )

            for r in results

        ]),

        3

    )

    # ==========================================
    # Output
    # ==========================================

    output = {

        "answer":
        answer,

        "sources":
        sources,

        "retrieval_score":
        retrieval_score

    }

    # ==========================================
    # Optional Context
    # ==========================================

    if return_context:

        output["context"] = (

            llm.build_context(
                results
            )

        )

    return output

In [ ]:
def print_rag_result(result):

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)

    print(result["answer"])

    print("\n" + "=" * 70)
    print("RETRIEVAL SCORE")
    print("=" * 70)

    print(result["retrieval_score"])

    print("\n" + "=" * 70)
    print("SOURCES")
    print("=" * 70)

    for idx, source in enumerate(
        result["sources"],
        start=1
    ):

        print(
            f"{idx}. "
            f"{source['source']} "
            f"(Page {source['page']}) "
            f"[Score: {source['score']}]"
        )

    print("=" * 70)

In [ ]:
result = rag_advanced(
    query="Summarize the findings of Lost in the Middle."
,
    retriever=retriever,
    llm=groq_llm,
    top_k=5,
    min_score=0,
    return_context=False
)

print_rag_result(result)